In [ ]:
import os
import pandas as pd
from multiprocessing import Pool, cpu_count
from functools import partial

def process_file(filename, ground_truth_folder, algorithm_folder):
    """
    Compare a single ground truth file with the corresponding algorithm result file.
    Returns a dict with 'query' and 'match' count.
    If algorithm file is missing or empty, match is 0.
    """
    ground_truth_file = os.path.join(ground_truth_folder, filename)
    algorithm_file = os.path.join(algorithm_folder, filename)

    # Read ground truth
    try:
        ground_truth_df = pd.read_csv(ground_truth_file)
    except Exception as e:
        print(f"[ERROR] Failed to read ground truth {filename}: {e}")
        return None  # Skip if ground truth missing/corrupt

    # If algorithm file is missing → 0 matches
    if not os.path.exists(algorithm_file):
        return {'query': filename, 'match': 0}

    # Process algorithm file
    try:
        algorithm_df = pd.read_csv(algorithm_file)
        algorithm_df = algorithm_df.drop_duplicates()
        ground_truth_first_10 = ground_truth_df.head(10)

        # Compute matches
        algorithm_ids = set(algorithm_df['ID'])
        ground_truth_ids = set(ground_truth_first_10['ID'])
        match_count = len(algorithm_ids.intersection(ground_truth_ids))

        return {'query': filename, 'match': match_count}

    except Exception as e:
        print(f"[ERROR] Failed to process algorithm file {filename}: {e}")
        return {'query': filename, 'match': 0}  # Treat errors as 0 matches

def process_folder(ground_truth_folder, algorithm_folder, output_file):
    """
    Process all files in a folder in parallel using multiprocessing.
    Returns a DataFrame with 'query' and 'match' for each file.
    """
    ground_truth_files = os.listdir(ground_truth_folder)

    # Ensure output directory exists
    output_directory = os.path.dirname(output_file)
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    # Partial function for multiprocessing
    process_file_partial = partial(
        process_file,
        ground_truth_folder=ground_truth_folder,
        algorithm_folder=algorithm_folder
    )

    # Use all CPU cores
    num_processes = cpu_count()
    with Pool(processes=num_processes) as pool:
        results = pool.map(process_file_partial, ground_truth_files)

    # Filter out files where ground truth could not be read
    results = [r for r in results if r is not None]

    # Save results
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_file, index=False)
    return results_df

def main(ground_truth_folder, algorithm_base_folder, output_base_folder, folder_suffixes):

    all_results = []
    result_strings = []

    existing_folders = os.listdir(algorithm_base_folder)

    for folder_suffix in folder_suffixes:
        folder_name = f'{folder_suffix}'

        if folder_name not in existing_folders:
            print(f"[INFO] Skipping {folder_name}, folder does not exist.")
            avg_zero = 0.0
            all_results.append({'folder': folder_suffix, 'average_match': avg_zero})
            result_strings.append(f"{avg_zero:.4f}")
            continue

        algorithm_folder = os.path.join(algorithm_base_folder, folder_name)
        output_file = os.path.join(output_base_folder, f'Res{folder_suffix}.csv')

        results_df = process_folder(ground_truth_folder, algorithm_folder, output_file)

        if results_df.empty:
            print(f"[INFO] No data in folder {folder_name}, treating all matches as 0.")
            avg_zero = 0.0
            all_results.append({'folder': folder_suffix, 'average_match': avg_zero})
            result_strings.append(f"{avg_zero:.4f}")
            continue

        if 'match' not in results_df.columns:
            print(f"[WARNING] 'match' column not found in results for folder {folder_name}, skipping.")
            continue

        average_match = results_df['match'].mean()

        all_results.append({'folder': folder_suffix, 'average_match': average_match})
        result_strings.append(f"{average_match/10:.4f}")

        print(f"[INFO] Processed {folder_name}: Average Match = {average_match:.2f}")

    os.makedirs(output_base_folder, exist_ok=True)

    summary_file = os.path.join(output_base_folder, 'SummaryResults.csv')
    pd.DataFrame(all_results).to_csv(summary_file, index=False)

    print(f"\nSummary written to {summary_file}")
    print("Result strings:", ", ".join(result_strings))

if __name__ == '__main__':
    ground_truth_folder = '/scratch/aa5f25/datasets/yt8m/Ground_truth/' 
    algorithm_base_folder = '/scratch/aa5f25/datasets/yt8m/Results/'
    output_base_folder = '/iridisfs/scratch/aa5f25/datasets/yt8m/IntermediateResults/'

    folder_suffixes = [20, 40, 80, 100, 200, 300, 500, 800, 1000, 1200, 1500, 1700, 1900, 2100, 2300] # These are the efs values

    main(
        ground_truth_folder,
        algorithm_base_folder,
        output_base_folder,
        folder_suffixes
    )
